# CodeGen — Group 45
## Step 2.5: Continued pre-training on monolingual Rust (the foundation)

**Why this step:** our diagnosis showed codegen-350M makes *language* mistakes (`.at()` instead
of indexing, wrong types) — it barely knows Rust. Before teaching it the *task* (Python→Rust),
we first teach it the *language* by training on lots of plain Rust code. No English, no Python —
just Rust, predicting the next token.

**What it produces:** a **Rust-aware base model** that Step 3 then fine-tunes on our validated
pairs. This gives the clean ablation: vanilla → **+monolingual** → +pairs.

**To run:** needs a **GPU** (`Runtime → Change runtime type → T4 GPU`), then `Runtime → Run all`.
This is scoped to run on free Colab (a bounded slice of Rust, a few hundred steps).


## 1. Install libraries
No Rust toolchain needed here — this step is pure text training (we don't compile anything).

In [ ]:
# Do NOT add `torch` (keeps Colab's torch/torchvision matched).
!pip install -q -U datasets transformers accelerate peft
# Colab's old torchao breaks newer peft; we don't use it.
!pip uninstall -q -y torchao
print("setup done")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.5 MB/s eta 0:00:00
setup done


## 2. Pull a scoped Rust corpus
We stream Rust files from `codeparrot/github-code` (filtered to Rust) and collect a bounded
number. We skip very large files to keep things manageable.

In [ ]:
from datasets import load_dataset

NUM_FILES = 2000        # bounded for free Colab; raise later if you have time
MAX_CHARS = 8000        # skip very large files

# Use ammarnasr/the-stack-rust-clean (parquet-based, no trust_remote_code, no auth needed)
# ~993k Rust files from The Stack corpus
stream = load_dataset("ammarnasr/the-stack-rust-clean", split="train")

texts = []
for ex in stream:
    code_str = ex["content"]
    if 50 <= len(code_str) <= MAX_CHARS:
        texts.append(code_str)
    if len(texts) >= NUM_FILES:
        break

print("collected", len(texts), "Rust files")
print("\n=== sample Rust file (head) ===\n", texts[0][:400])

# Old codeparrot/github-code approach (deprecated, uses loading script):
# stream = load_dataset("codeparrot/github-code", streaming=True, split="train",
#                       languages=["Rust"], trust_remote_code=True)
# texts = [ex["code"] for ex in stream][:NUM_FILES]

README.md:   0%|          | 0.00/1.71k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

data/train-00000-of-00024-12301fa50e8fb2(…):   0%|          | 0.00/137M [00:00<?, ?B/s]

data/train-00001-of-00024-1d2e41eb86ed9a(…):   0%|          | 0.00/138M [00:00<?, ?B/s]

data/train-00002-of-00024-e1f03c1a1d7a6f(…):   0%|          | 0.00/137M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/893792 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/49655 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/49656 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/22 [00:00<?, ?it/s]

collected 2000 Rust files

=== sample Rust file (head) ===
 #![allow(clippy::module_inception)]
#![allow(clippy::upper_case_acronyms)]
#![allow(clippy::large_enum_variant)]
#![allow(clippy::wrong_self_convention)]
#![allow(clippy::should_implement_trait)]
#![allow(clippy::blacklisted_name)]
#![allow(clippy::vec_init_then_push)]
#![allow(rustdoc::bare_urls)]
#![warn(missing_docs)]
//! <p></p>
//! <p>Amazon Managed Blockchain is a fully managed service for c


## 3. Tokenize and chunk into fixed-length blocks
For pre-training we concatenate all the Rust code and slice it into equal 512-token blocks —
the standard way to feed plain text to a causal language model.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

BASE = "Salesforce/codegen-350M-multi"
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token

BLOCK = 512
MAX_BLOCKS = 4000          # ~2M tokens — bounded; raise later for more training

ids = []
for t in texts:
    ids.extend(tok(t)["input_ids"] + [tok.eos_token_id])

blocks = [ids[i:i+BLOCK] for i in range(0, len(ids) - BLOCK, BLOCK)][:MAX_BLOCKS]
train_ds = Dataset.from_dict({"input_ids": blocks,
                              "attention_mask": [[1]*len(b) for b in blocks]})
print(len(blocks), "training blocks of", BLOCK, "tokens (~", len(blocks)*BLOCK//1000, "K tokens)")


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2218 > 2048). Running this sequence through the model will result in indexing errors


3397 training blocks of 512 tokens (~ 1739 K tokens)


## 4. Continued pre-training with LoRA
We train a small LoRA adapter on the Rust blocks — the model learns Rust syntax and idioms
while the 350M base stays frozen. A few hundred steps on a T4 takes ~15–30 min.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(BASE)
model.config.pad_token_id = tok.eos_token_id

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                  task_type="CAUSAL_LM", target_modules=["qkv_proj", "out_proj"])
model = get_peft_model(model, lora)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.print_trainable_parameters()

collator = DataCollatorForLanguageModeling(tok, mlm=False)
args = TrainingArguments(
    output_dir="ckpt_mono",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    save_strategy="no",
    report_to="none",
)
Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator).train()
print("\nmonolingual pre-training done")


pytorch_model.bin:   0%|          | 0.00/797M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-multi
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/797M [00:00<?, ?B/s]

trainable params: 1,966,080 || all params: 358,678,528 || trainable%: 0.5481


Step,Training Loss
20,1.760563
40,1.617533
60,1.550464
80,1.528617
100,1.445722
120,1.451374
140,1.447407
160,1.467987
180,1.454469
200,1.388507



monolingual pre-training done


## 5. Quick sanity check (optional)
Give it the start of a Rust function and see it continue — it should look more like real Rust now.

In [ ]:
model.eval()
prompt = "fn factorial(n: u64) -> u64 {\n"
inputs = tok(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=80, do_sample=False, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0], skip_special_tokens=True))


fn factorial(n: u64) -> u64 {
    let n = n;
    let sum = 0;
    for i in range(1, n + 1) {
        sum += i * i;
    }
    return sum;
}

fn factorial_n(n: u64) -> u64 {
    let n = n;
    let sum = 0;
    for i


## 6. Merge and save the Rust-aware base model
We bake the adapter into the weights (`merge_and_unload`) and save the result. **Step 3 then sets
`BASE` to this folder** so its fine-tune starts from a model that already knows Rust.

In [ ]:
merged = model.merge_and_unload()          # fold the LoRA into the base weights
merged.save_pretrained("codegen350m-rust-base")
tok.save_pretrained("codegen350m-rust-base")
print("saved Rust-aware base model to ./codegen350m-rust-base")

#Save to Drive so the team / Step 3 can reuse it across sessions:
from google.colab import drive; drive.mount("/content/drive")
import shutil; shutil.copytree("codegen350m-rust-base",
    "/content/drive/MyDrive/CodeGen_Group45/codegen350m-rust-base", dirs_exist_ok=True)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved Rust-aware base model to ./codegen350m-rust-base
Mounted at /content/drive


'/content/drive/MyDrive/CodeGen_Group45/codegen350m-rust-base'

## What we built (and how Step 3 uses it)
- A **Rust-aware base model** (`codegen350m-rust-base`) — codegen-350M after seeing lots of Rust.

**In Step 3, change one line:** set `BASE = "codegen350m-rust-base"` (or your Drive path) instead
of `"Salesforce/codegen-350M-multi"`. Everything else stays the same; you then fine-tune this
Rust-aware model on the validated pairs and re-evaluate.

**The ablation to report:** vanilla codegen-350M → +monolingual (this step) → +pairs (Step 3) →
+RAG. Each arrow is a number, and that progression is your core result.

**To scale:** raise `NUM_FILES` / `MAX_BLOCKS` and train longer for more Rust exposure — that, plus
the multi-sampled pairs from Step 2, is the main lever for moving the accuracy up.
